# 02 — Deteção de Recifes com ReefUNet (DGT HPC)

**Pipeline completo de inferência:** MDT-50cm mosaico → tiles 256×256 → ReefUNet → máscara recife → validação folium

Máquina: AMD EPYC-Milan 96 cores, 503 GB RAM, **sem GPU** (CPU-only)

| Passo | Script | Output |
|---|---|---|
| 1-4 | `01_lidar_hpc.ipynb` ✅ | `mosaico_algarve.tif` |
| 5 | `scripts/lidar/05_generate_tiles.py` | `tiles/tile_*.tif` |
| 6 | `scripts/dgt_inference_unet.py` | `masks/tile_*.tif` |
| 7 | `scripts/lidar/06_assemble_mask.py` | `recife_mask.tif` + `.gpkg` |
| 8 | Este notebook | Validação visual folium |

In [ ]:
import os, sys, json, logging
from pathlib import Path

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)-8s | %(name)s | %(message)s',
    handlers=[logging.StreamHandler(), logging.FileHandler('reef_pipeline.log', mode='a')],
)
logger = logging.getLogger('02_reef_detection')

# ── Paths ────────────────────────────────────────────────────────────────────
WORK         = Path('/home/jovyan/reef-imagery-pipeline')
OUTPUT_DIR   = Path('/home/jovyan/reef_output')
MODEL_PATH   = WORK / 'models' / 'unet_reef_best.pth'
MOSAIC_PATH  = OUTPUT_DIR / 'mosaico_algarve.tif'
MASK_PATH    = OUTPUT_DIR / 'recife_mask.tif'
PROB_PATH    = OUTPUT_DIR / 'recife_prob.tif'
GPKG_PATH    = OUTPUT_DIR / 'recife_mask.gpkg'

sys.path.insert(0, str(WORK))
os.chdir(str(WORK))

import torch
import numpy as np

print(f'PyTorch: {torch.__version__}  |  CPU threads: {torch.get_num_threads()}')
print(f'Mosaic:  {MOSAIC_PATH.exists()} — {MOSAIC_PATH}')
print(f'Model:   {MODEL_PATH.exists()} — {MODEL_PATH}')
print(f'Output:  {OUTPUT_DIR}')

In [ ]:
# ── PASSO 5 — Gerar tiles 256×256 com overlap 32px ───────────────────────────
from scripts.lidar.generate_tiles_05 import TileConfig, generate_tiles  # noqa: F401

# Importar directamente para evitar ambiguidade de nomes
import importlib, types
spec = importlib.util.spec_from_file_location(
    'gen_tiles', str(WORK / 'scripts/lidar/05_generate_tiles.py'))
mod_tiles = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod_tiles)

tile_cfg = mod_tiles.TileConfig(
    output_dir   = OUTPUT_DIR,
    mosaic_name  = 'mosaico_algarve.tif',
    tile_size    = 256,
    tile_overlap = 32,
    nodata_frac  = 0.80,
)

index_path = OUTPUT_DIR / 'tiles_index.json'
if index_path.exists():
    with open(str(index_path)) as f:
        existing = json.load(f)
    print(f'tiles_index.json já existe — {existing["n_tiles"]} tiles.  A saltar geração.')
    n_tiles = existing['n_tiles']
else:
    n_tiles = mod_tiles.generate_tiles(tile_cfg)
    print(f'Tiles gerados: {n_tiles}')

In [ ]:
# ── PASSO 6 — Inferência ReefUNet em batch ───────────────────────────────────
import importlib.util

spec = importlib.util.spec_from_file_location(
    'infer', str(WORK / 'scripts/dgt_inference_unet.py'))
mod_infer = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod_infer)

inference_log = OUTPUT_DIR / 'inference_log.json'
if inference_log.exists():
    with open(str(inference_log)) as f:
        prev = json.load(f)
    print(f'inference_log.json já existe — {prev["n_processed"]} tiles processados.')
    print(f'Reef fraction média: {prev["mean_reef_frac"]:.4f}  '
          f'Tiles com recife: {prev["tiles_with_reef"]}')
    print('Para re-processar: passar overwrite=True')
else:
    result = mod_infer.run_inference(
        output_dir = OUTPUT_DIR,
        model_path = MODEL_PATH,
        batch_size = 32,
        threshold  = 0.5,
        overwrite  = False,
    )
    print(f'Inferência concluída: {result["n_processed"]} tiles  {result["elapsed_s"]:.0f}s')
    print(f'Reef fraction média: {result["mean_reef_frac"]:.4f}')

In [ ]:
# ── PASSO 7 — Reconstituir mosaico de máscaras ───────────────────────────────
spec = importlib.util.spec_from_file_location(
    'assemble', str(WORK / 'scripts/lidar/06_assemble_mask.py'))
mod_assemble = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod_assemble)

if MASK_PATH.exists():
    print(f'recife_mask.tif já existe: {MASK_PATH}')
    import rasterio
    with rasterio.open(str(MASK_PATH)) as src:
        mask_arr = src.read(1)
        tags = src.tags()
    print(f'Reef fraction: {tags.get("reef_fraction", "?")}  '
          f'Reef pixels: {tags.get("reef_pixels", "?")}  '
          f'CRS: {src.crs}')
else:
    assemble_result = mod_assemble.assemble_mask(
        output_dir = OUTPUT_DIR,
        threshold  = 0.5,
        vectorize  = True,
    )
    print(f'Máscara gerada: {assemble_result["output_tif"]}')
    print(f'Reef fraction: {assemble_result["reef_fraction"]:.4f}  '
          f'({assemble_result["reef_pixels"]:,} pixels)')
    import rasterio
    with rasterio.open(str(MASK_PATH)) as src:
        mask_arr = src.read(1)

In [ ]:
# ── Estatísticas e distribuição de probabilidade ──────────────────────────────
import matplotlib.pyplot as plt

# Ler probabilidades
import rasterio
with rasterio.open(str(PROB_PATH)) as src:
    prob_arr = src.read(1)
    transform = src.transform
    crs = src.crs

valid_prob = prob_arr[prob_arr > 0].ravel()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].imshow(prob_arr, cmap='RdYlGn', vmin=0, vmax=1, interpolation='nearest')
axes[0].set_title('Probabilidade de Recife (continua)')
axes[0].axis('off')

axes[1].imshow(mask_arr, cmap='binary', interpolation='nearest')
axes[1].set_title(f'Máscara Binária (threshold=0.5)  —  {mask_arr.sum():,} px reef')
axes[1].axis('off')

axes[2].hist(valid_prob, bins=50, color='teal', alpha=0.8, edgecolor='k', linewidth=0.5)
axes[2].axvline(0.5, color='red', linestyle='--', label='threshold=0.5')
axes[2].set_title('Distribuição de Probabilidades')
axes[2].set_xlabel('P(recife)')
axes[2].set_ylabel('Nº pixels')
axes[2].legend()

plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / 'reef_mask_overview.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f'Overview guardado: {OUTPUT_DIR / "reef_mask_overview.png"}')

In [ ]:
# ── PASSO 8 — Validação visual com folium ────────────────────────────────────
import folium
import rasterio
from rasterio.warp import calculate_default_transform, reproject, Resampling
import tempfile, os

# Reprojectar máscara para WGS84 para folium
dst_crs = 'EPSG:4326'
with rasterio.open(str(MASK_PATH)) as src:
    src_crs = src.crs
    bounds_src = src.bounds
    tr, w, h = calculate_default_transform(src_crs, dst_crs, src.width, src.height, *bounds_src)
    meta = src.meta.copy()
    meta.update({'crs': dst_crs, 'transform': tr, 'width': w, 'height': h})

    tmp_path = str(OUTPUT_DIR / 'recife_mask_wgs84.tif')
    with rasterio.open(tmp_path, 'w', **meta) as dst:
        reproject(source=rasterio.band(src, 1), destination=rasterio.band(dst, 1),
                  src_transform=src.transform, src_crs=src_crs,
                  dst_transform=tr, dst_crs=dst_crs,
                  resampling=Resampling.nearest)

# Converter para imagem PNG para overlay folium
with rasterio.open(tmp_path) as src:
    mask_wgs = src.read(1)
    bounds_wgs = src.bounds  # (left, bottom, right, top)

# Criar mapa centrado na AOI Algarve
center_lat = (bounds_wgs.bottom + bounds_wgs.top) / 2
center_lon = (bounds_wgs.left  + bounds_wgs.right) / 2

m = folium.Map(location=[center_lat, center_lon], zoom_start=12,
               tiles='https://mt1.google.com/vt/lyrs=s&x={x}&y={y}&z={z}',
               attr='Google Satellite')

# Overlay: máscara binária como camada semi-transparente
rgba = np.zeros((*mask_wgs.shape, 4), dtype=np.uint8)
reef_pixels = mask_wgs == 1
rgba[reef_pixels, 0] = 0    # R
rgba[reef_pixels, 1] = 200  # G
rgba[reef_pixels, 2] = 100  # B
rgba[reef_pixels, 3] = 160  # Alpha

from PIL import Image
import base64, io
img = Image.fromarray(rgba, mode='RGBA')
buf = io.BytesIO()
img.save(buf, format='PNG')
img_b64 = base64.b64encode(buf.getvalue()).decode()

folium.raster_layers.ImageOverlay(
    image=f'data:image/png;base64,{img_b64}',
    bounds=[[bounds_wgs.bottom, bounds_wgs.left], [bounds_wgs.top, bounds_wgs.right]],
    opacity=0.7,
    name='Recife / Rocha (ReefUNet)',
).add_to(m)

# Adicionar GeoPackage vectorizado se existir
if GPKG_PATH.exists():
    try:
        import geopandas as gpd
        gdf = gpd.read_file(str(GPKG_PATH)).to_crs('EPSG:4326')
        folium.GeoJson(
            gdf.__geo_interface__,
            name='Recife (vectorizado)',
            style_function=lambda f: {'fillColor': '#00c864', 'color': '#007a3d',
                                       'weight': 1, 'fillOpacity': 0.4},
        ).add_to(m)
        logger.info('GeoPackage adicionado ao mapa: %d polígonos', len(gdf))
    except Exception as exc:
        logger.warning('Falha ao carregar GeoPackage para folium: %s', exc)

folium.LayerControl().add_to(m)

map_path = OUTPUT_DIR / 'reef_detection_map.html'
m.save(str(map_path))
print(f'Mapa guardado: {map_path}')

reef_pct = float(mask_wgs.mean()) * 100
print(f'\n=== RESUMO FINAL ===')
print(f'Área com recife/rocha: {reef_pct:.2f}% da AOI')
print(f'CRS original: {src_crs}')
print(f'GeoTIFF: {MASK_PATH}')
if GPKG_PATH.exists():
    print(f'GeoPackage: {GPKG_PATH}')
print(f'Mapa HTML: {map_path}')
m